In [ ]:
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
def explorer_dataset():
    X, y = load_breast_cancer(return_X_y=True)
    print(f"Lignes, colonnes : {X.shape}")
    valeurs, comptes = np.unique(y, return_counts=True)
    for v, c in zip(valeurs, comptes):
        label = "maligne" if v == 0 else "bénigne"
        print(f"Classe {v} ({label}) : {c} cas")
    return X, y

In [ ]:
X, y = explorer_dataset()

In [ ]:
#Phase 2 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [ ]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):

    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return accuracy_score(y_test, y_pred)

arbre = DecisionTreeClassifier(random_state=42)
acc = entrainer_et_evaluer(arbre, X_train, X_test, y_train, y_test)
print(f" Accuracy : {acc:.2%}")

In [ ]:
#Phase 3
def arene(X_train, X_test, y_train, y_test, titre="Breast Cancer"):
    import copy
    modeles = {
        "Arbre de décision"    : DecisionTreeClassifier(random_state=42),
        "Régression logistique": LogisticRegression(max_iter=10000, random_state=42),
        "KNN (k=5)"            : KNeighborsClassifier(n_neighbors=5),
    }
    
    resultats = {}
    for nom, modele in modeles.items():
        acc = entrainer_et_evaluer(copy.deepcopy(modele), X_train, X_test, y_train, y_test)
        resultats[nom] = acc
    
    resultats = dict(sorted(resultats.items(), key=lambda x: x[1], reverse=True))
    
    medailles = ["🥇", "🥈", "🥉", "4️⃣"]
    print(f"\n🏟️  Arène — {titre}")
    print(f"{'Rang':<5} {'Algorithme':<25} {'Accuracy':>10}")
    print("-" * 42)
    for i, (nom, acc) in enumerate(resultats.items()):
        print(f"{medailles[i]}    {nom:<25} {acc:>10.2%}")
    
    return resultats

resultats_cancer = arene(X_train, X_test, y_train, y_test)

In [ ]:
def clustering_aveugle(X, y):
  
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
   
    acc_direct  = accuracy_score(y, labels)
    acc_inverse = accuracy_score(y, 1 - labels)
    acc = max(acc_direct, acc_inverse)
    
    print("🔍 KMeans (non-supervisé) sans voir les étiquettes :")
    print(f"   Accuracy vs vraies classes : {acc:.2%}")
    print("\n   Répartition clusters / vraies classes :")
    data = load_breast_cancer()
    for i, nom_classe in enumerate(data.target_names):
        c0 = int(np.sum((y == i) & (labels == 0)))
        c1 = int(np.sum((y == i) & (labels == 1)))
        print(f"   {nom_classe:10s} → cluster 0 : {c0:3d}  |  cluster 1 : {c1:3d}")
    
    return labels

labels_kmeans = clustering_aveugle(X, y)